# Python 문법 파일별 3 · 05-1 구간 자르기 · 슬라이싱

**한 줄 요약** · `['e', 'f', 'g']`를 세 가지 슬라이싱으로 자르고, 시퀀스 공통 규칙과 학습 · 검증 나누기까지 정리

- **원본** · `1.Python문법/01_실습문제/05-1.Sequence Type Slicing_Q.ipynb` (원본 파일은 그대로 둠)
- **종합 정리본** · `00_최소선/01_Python문법` 전체 정리 2-2 (구간 자르기)

## 🎯 학습 목표

이 노트북을 끝내면 **자료를 보지 않고** 다음을 할 수 있어야 함

1. `[start:stop:step]`의 결과를 실행 전에 말한다 (stop은 포함 안 됨)
2. 같은 구간을 앞 번호 · 뒤 번호 두 방식으로 자른다
3. 인덱싱과 달리 슬라이싱은 범위 밖이어도 에러가 없음을 설명한다
4. 슬라이싱으로 학습 · 검증을 나누고 배치를 자른다

## 어디에 쓰나

*실제 상황 → 이 권에서 배우는 것*

- **학습 · 검증 데이터를 나눌 때** · `X[:n]` · `X[n:]` (3장)
- **데이터를 배치 크기로 자를 때** · `x[i:i + b]` (3장)
- **이미지 일부 · 최근 기록만 볼 때** · 앞 · 뒤 번호 슬라이싱 (1장 · 3장)
- **문자열 앞뒤를 자를 때** · 문자열도 같은 규칙 (2-1)

## 학습 지도

<img src="images/pf3_map.png" alt="05-1 학습 지도" width="763">

**예상 학습 시간** · 개념 40분 + 실습 20분

## 목차

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">장</th><th style="text-align: left;">제목</th><th style="text-align: left;">이 장의 질문</th></tr>
</thead>
<tbody>
<tr><td>0</td><td>준비</td><td>실행 환경 · 원본 문제</td></tr>
<tr><td>1</td><td>구간 자르기</td><td>연속된 구간을 어떻게 잘라내며, 끝 번호는 왜 포함되지 않는가?</td></tr>
<tr><td>2</td><td>시퀀스 공통 규칙</td><td>문자열 · 튜플에도 같은 규칙이 통하고, 슬라이스로 복사 · 교체는 어떻게 하는가?</td></tr>
<tr><td>3</td><td>종합 실습</td><td>샘플을 학습 · 검증으로 나누고 배치로 자르려면?</td></tr>
<tr><td>—</td><td>마무리 · 부록</td><td>오늘 배운 것 · 기억 확인 · 오답 노트 · 면접 · 요약표 · 복습일 · 원본 대응표 · 전체 코드</td></tr>
</tbody></table>

---
## 0. 준비

이 노트북은 라이브러리 없이 파이썬 기본 문법만 씀. **셀은 위에서부터 차례로 실행**(1-1의 `x`를 계속 씀).

*항목 → 설정*

- **라이브러리** · 없음
- **seed · 장치** · 필요 없음
- **데이터** · 원본 문제의 a ~ m 리스트 · 3장은 샘플 번호 0 ~ 9

**실행 흐름**

1. `sys` 모듈 불러오기
2. 파이썬 버전 출력
3. 3.10 이상인지 확인

In [1]:
# 실행 환경 확인: 이 노트북은 파이썬 기본 문법만 씀
# 왜 · 버전 차이로 출력이 달라지지 않는지 먼저 확인하기 위해
import sys  # 파이썬 버전 확인용

print("Python:", sys.version.split()[0])  # 3.10 이상이면 됨

Python: 3.14.6


### 원본 문제

- 아래 리스트에서 **슬라이싱**으로 `[e, f, g]`를 출력하는 문제
- 여러 슬라이싱 방법으로 시도

```python
x = ["a", "b", "c", "d", "e", "f", "g", "h", "i", "j", "k", "l", "m"]
```

**원본 힌트**

- 시퀀스 자료형 = 값이 순서대로 이어진 자료구조 (list · tuple · str · range)
- 리스트 슬라이싱 `L[start:stop:step]`
- 파이썬을 잘 쓰려면 리스트 자료구조가 매우 중요
- 원본은 리스트 연산마다 걸리는 시간(시간 복잡도) 참고 글도 소개

---
## 1. 구간 자르기

> **이 장의 흐름**
>
> - 원본 문제의 `[e, f, g]`를 세 가지로 자름
> - step · 생략 · 범위 밖까지 규칙을 넓힘

<img src="images/py1_slice.png" alt="4:7로 자르면 4 · 5 · 6번 (7번 미포함)" width="763">

**파이프라인 위치** · ③ 전처리 — 데이터 자르기, 학습 · 검증 나누기

### 1-1. `A` 세 가지 방법으로 자르기 · `x[a:b]` · `x[-a:-b]` <sub>원본 문제</sub>

> 🎯 **`['e', 'f', 'g']`를 어떤 슬라이스로 자르는가?**

💡 **쉽게 말하면**

- `[4:7]`은 **4번 사물함부터 7번 앞까지**.
- 끝을 빼야 `[0:4]` + `[4:8]`처럼 구간을 겹침 없이 이어 붙일 수 있음.

**실행 흐름**

1. 앞 번호 [4:7]
2. 뒤 번호 [-9:-6]
3. index()로 찾은 위치

**먼저 예측** · `x[4:7]`은 몇 개? — 답은 코드 실행 결과 아래에

In [2]:
x = ["a", "b", "c", "d", "e", "f", "g", "h", "i", "j", "k", "l", "m"]  # 번호 0 ~ 12


# 방법 1 · 2 · 3: 앞 번호 · 뒤 번호 · 위치를 찾아서
# 왜 · 같은 구간을 세 방식으로 잘라 끝 번호가 빠진다는 규칙을 확인하기 위해
print("앞 번호:", x[4:7])                            # 4, 5, 6 (7 미포함)
print("뒤 번호:", x[-9:-6])                          # 뒤에서 9번째 ~ 6번째 앞
print("index():", x[x.index("e"):x.index("g") + 1])  # g까지 넣으려면 +1

앞 번호: ['e', 'f', 'g']
뒤 번호: ['e', 'f', 'g']
index(): ['e', 'f', 'g']


**결과** · 예측 정답은 `3`개 — 4, 5, 6번

- 세 방법 모두 같은 결과
  - `앞 번호: ['e', 'f', 'g']`
  - `뒤 번호: ['e', 'f', 'g']`
  - `index(): ['e', 'f', 'g']`
- 길이 13에서 -9는 4번, -6은 7번 → `[-9:-6]` = `[4:7]`

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>x[i]</code></td><td>값 <strong>1개</strong> 반환</td><td>한 칸만 필요할 때</td></tr>
<tr><td><code>x[a:b]</code></td><td><strong>같은 자료형</strong>으로 구간 반환</td><td>구간 · 앞 n개 · 나누기</td></tr>
<tr><td><code>x[-a:-b]</code></td><td>뒤에서 센 구간</td><td>끝에서 몇 개 (길이 몰라도 됨)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - stop은 미포함 → 끝값까지 원하면 +1

> ✅ **한 줄 정리** · `[start:stop]`으로 자르며, stop은 포함되지 않아 개수 = stop - start다.

**딥러닝 코드에서는**

- `X_train, X_val = X[:800], X[800:]` — 학습 · 검증 나누기

### 1-2. `A` step · 생략 · 범위 밖 · `x[::step]` · `x[::-1]` <sub>원본 문제 · 여러 방법</sub>

> 🎯 **간격을 두거나 뒤집어 자르고, 범위를 넘으면 어떻게 되는가?**

💡 **쉽게 말하면**

- step은 몇 칸씩 건너뛸지, 음수면 거꾸로.
- 시작 · 끝을 비우면 처음 · 끝까지.

**실행 흐름**

1. 3칸씩
2. 뒤집기
3. 시작 · 끝 생략
4. 범위 밖 슬라이싱

**먼저 예측** · `x[10:20]`처럼 끝이 길이를 넘으면? — 답은 코드 실행 결과 아래에

In [3]:
# 왜 · step 값으로 건너뛰기 · 뒤집기 · 생략 규칙을 한 번에 비교하기 위해
print("3칸씩:", x[::3])        # 처음부터 끝까지 3칸씩
print("뒤집기:", x[::-1][:3])  # 뒤집은 뒤 앞 3개
print("생략:", x[:3], x[10:])  # 시작 생략 = 0, 끝 생략 = 끝까지
print("범위 밖:", x[10:20])    # 있는 만큼만, 에러 없음

3칸씩: ['a', 'd', 'g', 'j', 'm']
뒤집기: ['m', 'l', 'k']
생략: ['a', 'b', 'c'] ['k', 'l', 'm']
범위 밖: ['k', 'l', 'm']


**결과** · 예측 정답은 에러 없이 `['k', 'l', 'm']`까지만

- `3칸씩: ['a', 'd', 'g', 'j', 'm']` — step 3 → 3칸 간격
- `뒤집기: ['m', 'l', 'k']` — step -1 → 뒤집기
- `생략: ['a', 'b', 'c'] ['k', 'l', 'm']` — 생략하면 처음 · 끝
- `범위 밖: ['k', 'l', 'm']` — 인덱싱(IndexError)과 달리 슬라이싱은 범위 밖도 에러 없음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>x[::step]</code></td><td>간격 지정</td><td>n칸마다 고르기</td></tr>
<tr><td><code>x[::-1]</code></td><td>전체를 <strong>뒤집은</strong> 새 리스트</td><td>순서 뒤집기 (원본은 그대로)</td></tr>
<tr><td><code>x[:n]</code> · <code>x[n:]</code></td><td>앞 n개 · n번부터 끝</td><td>나누기 · 건너뛰기</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 범위 밖 슬라이싱은 에러가 안 나 실수를 놓치기 쉬움 → 길이 확인

> ✅ **한 줄 정리** · step으로 간격 · 방향을 정하고, 범위를 넘는 슬라이싱은 에러 없이 있는 만큼만 돌려준다.

**딥러닝 코드에서는**

- `img[:, ::2, ::2]` — 2칸마다 골라 크기 절반
- `seq[::-1]` — 시퀀스 뒤집기

---
## 2. 시퀀스 공통 규칙

> **이 장의 흐름**
>
> - 원본 힌트: 리스트 · 튜플 · 문자열 · range는 모두 시퀀스
> - 같은 규칙으로 자르고, 리스트는 슬라이스 자리에 값을 바꿔 넣을 수도 있음

<img src="images/pf3_seq.png" alt="시퀀스는 모두 같은 슬라이싱 규칙" width="763">

**파이프라인 위치** · ③ 전처리 — 문자열 · 튜플 다루기, 원본 보호

### 2-1. `B` 문자열 · 튜플도 같은 규칙 · `str[a:b]` · `tuple[a:b]` <sub>원본 힌트 · 시퀀스 자료형</sub>

> 🎯 **문자열 · 튜플을 자르면 결과는 어떤 자료형인가?**

💡 **쉽게 말하면**

- 리스트 · 문자열 · 튜플은 모두 **번호 붙은 줄**.
- 자르면 원래 자료형 그대로 나옴.

**실행 흐름**

1. 문자열 · 튜플 만들기
2. 같은 [4:7]
3. 자른 결과의 자료형

**먼저 예측** · 문자열 `s[4:7]`의 결과 자료형은? — 답은 코드 실행 결과 아래에

In [4]:
# 왜 · 리스트뿐 아니라 문자열 · 튜플도 같은 규칙으로 잘린다는 것을 보이기 위해
s = "abcdefghijklm"                                  # 문자열도 시퀀스
t = tuple(x)                                         # 튜플도 시퀀스
print(s[4:7], t[4:7])                                # 같은 규칙 → 같은 구간
print(type(s[4:7]).__name__, type(t[4:7]).__name__)  # 자른 결과는 원래 자료형
print(list(range(13))[4:7], range(13)[4:7])          # range도 시퀀스

efg ('e', 'f', 'g')
str tuple
[4, 5, 6] range(4, 7)


**결과** · 예측 정답은 `str` — 자르면 원래 자료형 그대로

- `efg ('e', 'f', 'g')` — 문자열 · 튜플도 같은 구간
- `str tuple` — 자른 결과는 원래 자료형
- `[4, 5, 6] range(4, 7)` — range를 자르면 range

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>list[a:b]</code></td><td>리스트 반환</td><td>값을 바꿀 수 있어야 할 때</td></tr>
<tr><td><code>str[a:b]</code></td><td>문자열 반환</td><td>앞 · 뒤 글자 자르기</td></tr>
<tr><td><code>tuple[a:b]</code></td><td>튜플 반환 (바꿀 수 없음)</td><td>고정된 묶음 (shape 등)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 문자열 · 튜플은 자를 수는 있어도 `s[0] = "A"`처럼 자리를 바꿀 수는 없음 (TypeError)

> ✅ **한 줄 정리** · 리스트 · 문자열 · 튜플 · range는 모두 같은 슬라이싱 규칙을 쓰고, 결과는 원래 자료형이다.

**딥러닝 코드에서는**

- `shape[1:]` — 튜플 shape에서 배치 차원을 뺀 나머지
- `name[:-4]` — 파일 이름에서 `.jpg` 떼기

### 2-2. `B` 슬라이스로 복사 · 교체 · 삭제 · `x[:]` · `x[a:b] = ...` · `del x[a:]` <sub>보충 · 리스트 슬라이스</sub>

> 🎯 **`x[:]`는 무엇을 만들고, 슬라이스 자리에 값을 넣으면 어떻게 되는가?**

💡 **쉽게 말하면**

- `x[:]`는 처음부터 끝까지 잘라낸 **새 리스트** = 복사.
- 리스트는 잘라낸 자리에 다른 값을 넣거나 지울 수 있음.

**실행 흐름**

1. x[:]로 복사 후 수정
2. 구간 교체
3. 구간 삭제

**먼저 예측** · `w = x[:]` 뒤 `w[0] = "A"` 하면 `x[0]`은? — 답은 코드 실행 결과 아래에

In [5]:
# 왜 · [:] 슬라이싱이 새 리스트를 만든다는 것을 원본과 비교해 확인하기 위해
w = x[:]                                                       # 처음부터 끝까지 = 얕은 복사
w[0] = "A"                                                     # 복사본만 수정
print("x[0]:", x[0], "/ w[0]:", w[0], "/ 같은 객체?", w is x)  # 원본 그대로


# 왜 · 구간 대입 · 삭제를 원본이 아닌 사본에서 실험하기 위해
v = x.copy()           # 바꾸기 실습용
v[1:3] = ["B", "C"]    # 구간을 다른 값으로 교체
print("교체:", v[:4])  # 1 · 2번이 바뀜
del v[3:]              # 3번부터 끝까지 삭제
print("삭제 후:", v)   # 앞 3개만 남음

x[0]: a / w[0]: A / 같은 객체? False
교체: ['a', 'B', 'C', 'd']
삭제 후: ['a', 'B', 'C']


**결과** · 예측 정답은 `a` 그대로 — `x[:]`는 새 리스트

- `x[0]: a / w[0]: A / 같은 객체? False` — 슬라이스는 새 객체 → 원본 보존
- `교체: ['a', 'B', 'C', 'd']` — 1 · 2번 구간을 새 값으로
- `삭제 후: ['a', 'B', 'C']` — `del v[3:]` → 3번부터 끝까지 삭제

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>x[:]</code></td><td>전체 얕은 복사</td><td><code>x.copy()</code>와 같은 결과</td></tr>
<tr><td><code>x[a:b] = [...]</code></td><td>구간을 <strong>바꿔 끼움</strong></td><td>일부를 한 번에 교체</td></tr>
<tr><td><code>del x[a:b]</code></td><td>구간 <strong>삭제</strong></td><td>앞 n개만 남기기</td></tr>
<tr><td><code>x[a:b]</code> (읽기)</td><td>새 리스트, 원본 그대로</td><td>자르기만 할 때</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - `x[:]`도 얕은 복사 → 리스트 안 리스트는 안쪽을 공유 (파일별 1의 2-2)

> ✅ **한 줄 정리** · `x[:]`는 새 리스트를 만들고, 리스트는 슬라이스 자리에 값을 바꿔 넣거나 `del`로 지울 수 있다.

**딥러닝 코드에서는**

- `history = history[-100:]` — 최근 100개만 남기기

---
## 3. 종합 실습 · 학습 · 검증 나누기와 배치

> **이 장의 흐름**
>
> - 1 · 2장 문법을 한 번에 사용
> - 딥러닝 학습 전 단계인 데이터 나누기 · 배치 자르기를 슬라이싱만으로

<img src="images/pf3_practice.png" alt="샘플 10개를 8 · 2로 나누고 학습 8개를 3개씩 배치로" width="763">

**파이프라인 위치** · ① 데이터 준비 → ③ 전처리 → ⑤ 학습

> 🎯 **샘플 10개를 80% 학습 · 20% 검증으로 겹치지 않게 나누고, 학습 데이터를 3개씩 배치로 자르려면?**

💡 **쉽게 말하면**

- `[:n]`과 `[n:]`은 **같은 n에서 끊어 이어 붙이면 원본**.
- 배치는 `[i:i + b]`를 b칸씩 옮기며 자르기.

**실행 흐름**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">단계</th><th style="text-align: left;">하는 일</th><th style="text-align: left;">쓰는 문법</th></tr>
</thead>
<tbody>
<tr><td>1</td><td>샘플 번호 0 ~ 9</td><td>range · list</td></tr>
<tr><td>2</td><td>8 · 2로 나누기</td><td>1-1 <code>[:n]</code> · <code>[n:]</code></td></tr>
<tr><td>3</td><td>3개씩 배치</td><td>1-2 step · <code>range(0, n, b)</code></td></tr>
<tr><td>4</td><td>최근 3개</td><td>1-1 뒤 번호</td></tr>
</tbody></table>

**먼저 예측** · 학습 8개를 3개씩 자르면 배치는 몇 개이고 마지막 배치는 몇 개일까? — 답은 코드 실행 결과 아래에

In [6]:
# 1단계: 샘플 10개 (번호로 대신)
# 왜 · 학습 데이터를 나누는 실제 상황을 작은 번호 목록으로 만들기 위해
samples = list(range(10))  # 0 ~ 9


# 2단계: 80% 학습 · 20% 검증으로 나누기
# 왜 · 같은 경계 번호로 잘라 학습 · 검증이 겹치지 않게 하기 위해
n_train = int(len(samples) * 0.8)                  # 학습 개수
train, val = samples[:n_train], samples[n_train:]  # 앞 n개 · 나머지
print("train:", train, "/ val:", val)              # 겹침 없이 나뉨
print("빠진 것 없음:", train + val == samples)     # 이어 붙이면 원본


# 3단계: 학습 데이터를 3개씩 배치로 자르기
# 왜 · range의 step과 슬라이싱으로 마지막 짧은 배치까지 빠짐없이 자르기 위해
batch_size = 3                               # 배치 크기
for i in range(0, len(train), batch_size):   # 0, 3, 6
    print("배치:", train[i:i + batch_size])  # 마지막 배치는 남은 만큼


# 4단계: 시계열처럼 최근 3개만 보기
# 왜 · 음수 번호로 길이를 몰라도 최근 값만 꺼내기 위해
print("최근 3개:", samples[-3:])  # 뒤에서 3개

train: [0, 1, 2, 3, 4, 5, 6, 7] / val: [8, 9]
빠진 것 없음: True
배치: [0, 1, 2]
배치: [3, 4, 5]
배치: [6, 7]
최근 3개: [7, 8, 9]


**결과** · 예측 정답은 배치 3개 · 마지막은 2개

- `train: [0, 1, 2, 3, 4, 5, 6, 7] / val: [8, 9]` — 10 × 0.8 = 8 → 앞 8개 · 나머지 2개
- `빠진 것 없음: True` — `[:n]` + `[n:]` = 원본 (stop 미포함 덕분)
- 범위 밖도 에러 없이 남은 2개
  - `배치: [0, 1, 2]`
  - `배치: [3, 4, 5]`
  - `배치: [6, 7]`
- `최근 3개: [7, 8, 9]` — 뒤에서 3개

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>X[:n]</code> · <code>X[n:]</code> (순서대로)</td><td>앞 · 뒤로 끊음</td><td>시계열처럼 순서가 의미 있을 때</td></tr>
<tr><td>섞은 뒤 <code>[:n]</code> · <code>[n:]</code></td><td>무작위 분할</td><td>이미지 · 문장처럼 순서가 의미 없을 때</td></tr>
<tr><td><code>x[i:i + b]</code> 반복</td><td>배치를 직접 자름</td><td>원리 확인 · 작은 데이터</td></tr>
<tr><td><code>DataLoader(batch_size=b)</code></td><td>배치를 자동으로</td><td>PyTorch 학습 (기본 선택)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 순서대로 끊으면 클래스가 한쪽에 몰릴 수 있음 → 이미지 분류는 섞은 뒤 나눔 (시계열은 순서 유지)

> ✅ **한 줄 정리** · `[:n]`과 `[n:]`으로 겹침 없이 나누고, `[i:i + b]`를 b칸씩 옮기며 배치를 자른다.

**딥러닝 코드에서는**

- `train_set, val_set = data[:n], data[n:]`
- `DataLoader`가 내부에서 하는 일이 바로 3단계

---
## 그래서 오늘 무엇을 배웠는가?

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">학습 목표</th><th style="text-align: left;">할 수 있게 된 것</th><th style="text-align: left;">확인 위치</th></tr>
</thead>
<tbody>
<tr><td>1. stop 미포함</td><td><code>x[4:7]</code>은 3개, <code>[0:4] + [4:8]</code>은 겹침 없이 이어짐</td><td>1-1</td></tr>
<tr><td>2. 앞 · 뒤 번호</td><td><code>x[4:7]</code> = <code>x[-9:-6]</code></td><td>1-1</td></tr>
<tr><td>3. 범위 밖</td><td><code>x[10:20]</code>은 에러 없이 <code>['k', 'l', 'm']</code></td><td>1-2</td></tr>
<tr><td>4. 나누기 · 배치</td><td>10개 → 8 · 2, 학습 8개 → 3 · 3 · 2</td><td>3</td></tr>
</tbody></table>

### 주제별 한 줄 · 반드시 기억할 것

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">등급</th><th style="text-align: left;">주제</th><th style="text-align: left;">반드시 기억할 것</th><th style="text-align: left;">딥러닝 코드에서</th></tr>
</thead>
<tbody>
<tr><td><code>A</code></td><td>슬라이싱</td><td><code>[start:stop:step]</code>, stop 미포함</td><td><code>X[:800]</code> · <code>X[800:]</code></td></tr>
<tr><td><code>A</code></td><td>범위 밖</td><td>슬라이싱은 에러 없음</td><td>마지막 배치는 남은 만큼</td></tr>
<tr><td><code>B</code></td><td>시퀀스 공통</td><td>문자열 · 튜플 · range도 같은 규칙</td><td><code>shape[1:]</code></td></tr>
<tr><td><code>B</code></td><td>슬라이스 복사 · 교체</td><td><code>x[:]</code> · <code>x[a:b] = ...</code> · <code>del</code></td><td><code>history[-100:]</code></td></tr>
</tbody></table>

---
## 기억 확인 문제

위 내용을 **보지 않고** 답한 뒤, ✅ 한 줄 정리와 비교해 ○ (정확) · △ (일부) · × (모름)를 표시. △ · ×인 섹션만 다시 보기.

### 1. 질문에 답하기

*섹션 → 🎯 질문*

- **1-1** · `x[4:7]`의 개수와 7번이 빠지는 이유는?
- **1-2** · 슬라이싱이 범위를 넘으면?
- **2-1** · 문자열을 자르면 결과 자료형은?
- **2-2** · `x[:]`는 무엇을 만드는가?
- **3** · 10개를 8 · 2로 나누는 코드는?

#### 내 답 · 이 셀을 더블클릭해서 작성

**1-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

### 2. 코드로 답하기

아래 빈 셀에 직접 작성 후 실행.

1. `x`에서 `['h', 'i', 'j']`를 앞 번호로 자르기
2. `x`의 짝수 번호(0, 2, 4, …)만 꺼내기

In [7]:
# 코드 문제 직접 작성

### 3. 변형 문제 · 조건을 바꿔 다시 풀기

1. `x`의 마지막 4개를 뒤 번호로 자르기
2. `samples = list(range(12))`를 75% · 25%로 나누면 각각 몇 개?
3. `"image_001.jpg"`에서 확장자 `.jpg`를 뺀 이름만 슬라이싱으로 꺼내기

In [8]:
# 변형 문제 직접 작성

### 4. 모범 답안 · 1 ~ 3을 다 푼 뒤에 보기

#### 질문 답

*섹션 → 답*

- **1-1** · 3개 · 끝을 빼야 구간을 겹침 없이 이어 붙일 수 있음
- **1-2** · 에러 없이 있는 만큼만 (인덱싱은 IndexError)
- **2-1** · 문자열 (원래 자료형 그대로)
- **2-2** · 새 리스트 (얕은 복사)
- **3** · `train, val = s[:8], s[8:]`

#### 코드 문제 답

```python
x = ["a", "b", "c", "d", "e", "f", "g", "h", "i", "j", "k", "l", "m"]
print(x[7:10])  # ['h', 'i', 'j']
print(x[::2])  # ['a', 'c', 'e', 'g', 'i', 'k', 'm']
```

#### 변형 문제 답

*문제 → 답*

- **1** · `x[-4:]` → `['j', 'k', 'l', 'm']`
- **2** · 12 × 0.75 = 9 → 학습 9개 · 검증 3개
- **3** · `name[:-4]` → `image_001`

---
## 오답 노트

틀린 문제는 정답을 베끼지 말고 **원인**을 적음. 다음 복습 때 이 표의 "재발 방지 질문"부터 스스로에게 물어보기.

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">틀린 내용</th><th style="text-align: left;">틀린 이유</th><th style="text-align: left;">올바른 원칙</th><th style="text-align: left;">재발 방지 질문</th></tr>
</thead>
<tbody>
<tr><td>(예) <code>x[4:7]</code>을 4개로 답함</td><td>끝 번호 7도 들어간다고 착각</td><td>stop은 미포함 → 7 - 4 = 3개</td><td>끝 번호를 빼고 셌나?</td></tr>
<tr><td></td><td></td><td></td><td></td></tr>
<tr><td></td><td></td><td></td><td></td></tr>
</tbody></table>

---
## 면접 30초 답변

**Q1. 슬라이싱에서 끝 번호를 포함하지 않는 이유는 무엇인가요?**
> - `x[:n]`과 `x[n:]`을 같은 n으로 이어 붙이면 원본이 되고, 개수가 `stop - start`로 바로 계산되기 때문입니다.
> - 학습 · 검증을 겹침 없이 나눌 때 이 성질을 그대로 씁니다.

**Q2. 인덱싱과 슬라이싱은 범위를 넘을 때 어떻게 다른가요?**
> - 인덱싱 `x[i]`는 없는 번호면 `IndexError`가 나지만, 슬라이싱 `x[a:b]`는 있는 만큼만 돌려주고 에러가 없습니다.
> - 그래서 마지막 배치가 작아도 에러 없이 처리되지만, 실수도 조용히 넘어가므로 길이를 확인해야 합니다.

**Q3. 학습 · 검증 데이터는 어떻게 나누나요?**
> - 개수 n을 정한 뒤 `data[:n]`과 `data[n:]`으로 겹침 없이 나눕니다.
> - 이미지처럼 순서가 의미 없는 데이터는 먼저 섞고 나누며, 시계열은 미래 정보가 섞이지 않도록 순서대로 나눕니다.

---
## 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>x[a:b]</code> · <code>x[-a:-b]</code></td><td>Python 기본 문법</td><td>구간 자르기</td><td>1-1 <code>x[4:7]</code> · <code>x[-9:-6]</code></td></tr>
<tr><td><code>x[::step]</code> · <code>x[::-1]</code></td><td>Python 기본 문법</td><td>간격 · 뒤집기</td><td>1-2 <code>x[::3]</code> · <code>x[::-1]</code></td></tr>
<tr><td><code>tuple()</code> · <code>range()</code></td><td>Python 내장 함수</td><td>튜플 · 숫자 범위 (시퀀스)</td><td>2-1 <code>tuple(x)</code> · <code>range(13)[4:7]</code></td></tr>
<tr><td><code>x[:]</code> · <code>x[a:b] = ...</code> · <code>del x[a:]</code></td><td>list</td><td>복사 · 교체 · 삭제</td><td>2-2 <code>v[1:3] = ["B", "C"]</code></td></tr>
<tr><td><code>int()</code></td><td>Python 내장 함수</td><td>개수를 정수로</td><td>3 <code>int(len(samples) * 0.8)</code></td></tr>
<tr><td><code>range(0, n, b)</code></td><td>Python 내장 함수</td><td>배치 시작 번호</td><td>3 <code>range(0, len(train), batch_size)</code></td></tr>
</tbody></table>

### 할 것 · 하지 말 것

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">✅ 할 것</th><th style="text-align: left;">⚠️ 하지 말 것</th></tr>
</thead>
<tbody>
<tr><td>끝값까지 원하면 stop + 1</td><td>stop을 포함된다고 세기</td></tr>
<tr><td>나눈 뒤 <code>train + val == data</code> 확인</td><td>범위 밖 슬라이싱을 믿고 길이 확인 생략</td></tr>
<tr><td>이미지 분류는 섞고 나누기</td><td>시계열을 섞어서 나누기</td></tr>
<tr><td>원본 보호는 <code>x[:]</code> · <code>copy()</code></td><td>문자열 자리에 대입 (<code>s[0] = ...</code>)</td></tr>
</tbody></table>

### 헷갈리는 짝 한눈에 보기

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">짝</th><th style="text-align: left;">차이</th><th style="text-align: left;">결론</th></tr>
</thead>
<tbody>
<tr><td><code>x[i]</code> vs <code>x[a:b]</code></td><td>값 1개 vs 같은 자료형</td><td>구간은 슬라이싱</td></tr>
<tr><td>인덱싱 vs 슬라이싱 범위 밖</td><td>IndexError vs 에러 없음</td><td>슬라이싱은 길이 확인</td></tr>
<tr><td><code>x[::-1]</code> vs <code>x.reverse()</code></td><td>새 리스트 vs 원본 뒤집기</td><td>원본을 지킬 땐 <code>[::-1]</code></td></tr>
<tr><td><code>x[:]</code> vs <code>y = x</code></td><td>새 리스트 vs 같은 객체</td><td>복사는 <code>x[:]</code></td></tr>
</tbody></table>

---
## 다음 복습일

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">언제</th><th style="text-align: left;">할 일</th><th style="text-align: left;">걸리는 시간</th></tr>
</thead>
<tbody>
<tr><td>내일</td><td>기억 확인 문제 1을 자료 없이 다시 답하고 오답 노트 갱신</td><td>10분</td></tr>
<tr><td>3일 뒤</td><td>변형 문제 3개 다시 풀기 · 1장 결과를 손으로 먼저 적기</td><td>15분</td></tr>
<tr><td>7일 뒤</td><td>3장을 샘플 20개 · 배치 4로 바꿔 다시 짜기</td><td>20분</td></tr>
<tr><td>14일 뒤</td><td>오답 노트에 남은 문제만 · 파일별 4(15-1) 시작 전 ✅ 줄만 읽기</td><td>10분</td></tr>
</tbody></table>

---
## 부록

### 이 노트북을 만든 방법

- 1장 · 원본 문제(`05-1.Sequence Type Slicing_Q.ipynb`)의 여러 슬라이싱 방법
- 2장 · 힌트의 시퀀스 자료형을 문자열 · 튜플 · 복사로 확장
- 모든 코드는 실행한 출력과 함께 저장 · 형식 v2.1 (FORMAT_V2.md 1 ~ 7절)

### 읽는 법

*표시 → 의미*

- **`A` · `B`** · **A** 딥러닝 코드에 바로 나오는 필수 · **B** 알아두면 되는 것
- **🎯** · 이번 섹션에서 답해야 할 질문
- **💡** · 쉬운 설명 · 비유
- **⚠️** · 자주 하는 실수
- **✅** · 기억할 결론 = 🎯 질문의 답

- **섹션 앞부분** · 🎯 질문 → 💡 쉬운 설명 → 실행 흐름 → 먼저 예측 → 코드
- **섹션 뒷부분** · 결과(예측 정답 · 출력 줄) → 차이 · 언제 쓰나 → ⚠️ 주의 → ✅ 한 줄 정리
- **코드 주석** · 코드 위는 이 단계에서 하는 일, 코드 옆은 그 줄이 필요한 이유 (줄마다 · 열 맞춤)

### 원본 → 섹션 대응표

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">원본</th><th style="text-align: left;">내용</th><th style="text-align: left;">이 노트북</th></tr>
</thead>
<tbody>
<tr><td>문제 · 출력 결과 [e, f, g]</td><td>세 가지 방법으로 자르기</td><td>1-1</td></tr>
<tr><td>문제 · 다양한 방법</td><td>step · 생략 · 범위 밖</td><td>1-2</td></tr>
<tr><td>힌트 · 시퀀스 자료형</td><td>문자열 · 튜플 · range</td><td>2-1</td></tr>
<tr><td>힌트 · 리스트 자료구조 중요</td><td>슬라이스 복사 · 교체 · 삭제</td><td>2-2</td></tr>
<tr><td>힌트 · 시간 복잡도 참고 글</td><td>0장 원본 힌트에 소개만</td><td>0장</td></tr>
</tbody></table>

### 전체 코드

1 ~ 3장 코드를 장별 셀로 모음. **위에서부터 이 셀들만 실행해도 전체가 동작**하며, 각 블록 머리 주석의 번호가 섹션.

**실행 흐름**

1. 1-1 앞 번호 · 뒤 번호로 구간 자르기
2. 1-2 step · 뒤집기 · 생략
3. 블록마다 잘라 낸 값 출력

In [9]:
# ---------- 1-1 ----------
x = ["a", "b", "c", "d", "e", "f", "g", "h", "i", "j", "k", "l", "m"]  # 번호 0 ~ 12


# 방법 1 · 2 · 3: 앞 번호 · 뒤 번호 · 위치를 찾아서
# 왜 · 같은 구간을 세 방식으로 잘라 끝 번호가 빠진다는 규칙을 확인하기 위해
print("앞 번호:", x[4:7])                            # 4, 5, 6 (7 미포함)
print("뒤 번호:", x[-9:-6])                          # 뒤에서 9번째 ~ 6번째 앞
print("index():", x[x.index("e"):x.index("g") + 1])  # g까지 넣으려면 +1


# ---------- 1-2 ----------
# 왜 · step 값으로 건너뛰기 · 뒤집기 · 생략 규칙을 한 번에 비교하기 위해
print("3칸씩:", x[::3])        # 처음부터 끝까지 3칸씩
print("뒤집기:", x[::-1][:3])  # 뒤집은 뒤 앞 3개
print("생략:", x[:3], x[10:])  # 시작 생략 = 0, 끝 생략 = 끝까지
print("범위 밖:", x[10:20])    # 있는 만큼만, 에러 없음

앞 번호: ['e', 'f', 'g']
뒤 번호: ['e', 'f', 'g']
index(): ['e', 'f', 'g']
3칸씩: ['a', 'd', 'g', 'j', 'm']
뒤집기: ['m', 'l', 'k']
생략: ['a', 'b', 'c'] ['k', 'l', 'm']
범위 밖: ['k', 'l', 'm']


**실행 흐름**

1. 2-1 문자열 · 튜플 슬라이싱
2. 2-2 `[:]` 복사와 구간 대입 · 삭제
3. 블록마다 원본 상태 출력

In [10]:
# ---------- 2-1 ----------
# 왜 · 리스트뿐 아니라 문자열 · 튜플도 같은 규칙으로 잘린다는 것을 보이기 위해
s = "abcdefghijklm"                                  # 문자열도 시퀀스
t = tuple(x)                                         # 튜플도 시퀀스
print(s[4:7], t[4:7])                                # 같은 규칙 → 같은 구간
print(type(s[4:7]).__name__, type(t[4:7]).__name__)  # 자른 결과는 원래 자료형
print(list(range(13))[4:7], range(13)[4:7])          # range도 시퀀스


# ---------- 2-2 ----------
# 왜 · [:] 슬라이싱이 새 리스트를 만든다는 것을 원본과 비교해 확인하기 위해
w = x[:]                                                       # 처음부터 끝까지 = 얕은 복사
w[0] = "A"                                                     # 복사본만 수정
print("x[0]:", x[0], "/ w[0]:", w[0], "/ 같은 객체?", w is x)  # 원본 그대로


# 왜 · 구간 대입 · 삭제를 원본이 아닌 사본에서 실험하기 위해
v = x.copy()           # 바꾸기 실습용
v[1:3] = ["B", "C"]    # 구간을 다른 값으로 교체
print("교체:", v[:4])  # 1 · 2번이 바뀜
del v[3:]              # 3번부터 끝까지 삭제
print("삭제 후:", v)   # 앞 3개만 남음

efg ('e', 'f', 'g')
str tuple
[4, 5, 6] range(4, 7)
x[0]: a / w[0]: A / 같은 객체? False
교체: ['a', 'B', 'C', 'd']
삭제 후: ['a', 'B', 'C']


**실행 흐름**

1. 샘플 10개 준비
2. 80% · 20%로 나누기
3. 3개씩 배치로 자르기
4. 최근 3개 출력

In [11]:
# ---------- 3. 종합 실습 ----------
# 1단계: 샘플 10개 (번호로 대신)
# 왜 · 학습 데이터를 나누는 실제 상황을 작은 번호 목록으로 만들기 위해
samples = list(range(10))  # 0 ~ 9


# 2단계: 80% 학습 · 20% 검증으로 나누기
# 왜 · 같은 경계 번호로 잘라 학습 · 검증이 겹치지 않게 하기 위해
n_train = int(len(samples) * 0.8)                  # 학습 개수
train, val = samples[:n_train], samples[n_train:]  # 앞 n개 · 나머지
print("train:", train, "/ val:", val)              # 겹침 없이 나뉨
print("빠진 것 없음:", train + val == samples)     # 이어 붙이면 원본


# 3단계: 학습 데이터를 3개씩 배치로 자르기
# 왜 · range의 step과 슬라이싱으로 마지막 짧은 배치까지 빠짐없이 자르기 위해
batch_size = 3                               # 배치 크기
for i in range(0, len(train), batch_size):   # 0, 3, 6
    print("배치:", train[i:i + batch_size])  # 마지막 배치는 남은 만큼


# 4단계: 시계열처럼 최근 3개만 보기
# 왜 · 음수 번호로 길이를 몰라도 최근 값만 꺼내기 위해
print("최근 3개:", samples[-3:])  # 뒤에서 3개

train: [0, 1, 2, 3, 4, 5, 6, 7] / val: [8, 9]
빠진 것 없음: True
배치: [0, 1, 2]
배치: [3, 4, 5]
배치: [6, 7]
최근 3개: [7, 8, 9]
